In [1]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: /var/folders/dd/tpdzgsps29l6xqx8hm7z2_rr0000gp/T/tmpeq45vxhn


In [2]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load env vars

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
from anthropic.types import MessageParam
from collections.abc import Iterable

def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text }
    messages.append(assistant_message)

from anthropic.types import TextBlock, Message

def get_message_text(message: Message) -> str:
    return next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "" # empty string if none
    )

def chat(messages: Iterable[MessageParam], system: str|None = None, temperature = 1.0, stop_sequences: list[str] = []) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_message_text(message)

In [6]:
import json
from typing import TypedDict, Literal

class TaskObj(TypedDict):
    task: str
    format: Literal["python", "json", "regex"]

class GradedTestCaseObj(TypedDict):
    test_case: TaskObj
    output: str
    model_score: int
    syntax_score: int
    average_score: float
    reasoning: str

with open('data/009_generate_eval_dataset-tasks.json', 'r', encoding='utf-8') as file:
    dataset: list[TaskObj] = json.load(file)

print(dataset)

[{'task': "Parse an AWS S3 bucket URI and extract the bucket name and object key from a string like 's3://my-bucket/path/to/object.txt'", 'format': 'regex'}, {'task': "Create a JSON policy document that grants an IAM user read-only access to a specific S3 bucket named 'company-data'", 'format': 'json'}, {'task': "Write a Python function that takes an AWS region code (e.g., 'us-east-1') and returns the corresponding region name (e.g., 'N. Virginia')", 'format': 'python'}]


In [7]:
import ast
import re

def validate_json(text: str):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text: str):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text: str):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response: str, test_case: TaskObj):
    format = test_case['format']
    if format == "json":
        return validate_json(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        return validate_python(response)

In [8]:
def run_prompt(test_case: TaskObj) -> str:
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output


In [9]:
class EvalReturnObj(TypedDict):
    score: int
    reasoning: str

def grade_by_model(test_case: TaskObj, output: str) -> EvalReturnObj:
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task:
    <task>
    {test_case["task"]}
    </task>

    Solution:
    <solution>
    {output}
    </solution>
    
    Provide your evaluation as a structured JSON object like this:
    ```json
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    ```
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    res: EvalReturnObj = json.loads(eval_text)
    return res


In [ ]:
def run_test_case(test_case: TaskObj) -> GradedTestCaseObj:
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)

    syntax_score = grade_syntax(output, test_case)

    result: GradedTestCaseObj = {
        "output": output,
        "test_case": test_case,
        "model_score": model_grade["score"],
        "syntax_score": syntax_score,
        "average_score": (model_grade["score"] + syntax_score) / 2,
        "reasoning": model_grade["reasoning"]
    }
    
    return result

from statistics import mean

def run_eval(dataset: Iterable[TaskObj]) -> list[GradedTestCaseObj]:
    """Loads the dataset and calls run_test_case with each case"""
    results: list[GradedTestCaseObj] = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["average_score"] for result in results])

    print(f"Average score: {average_score}")
    
    return results

In [11]:
results = run_eval(dataset)

In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef parse_s3_uri(uri):\n    match = re.match(r'^s3://([a-z0-9.-]+)/(.+)$', uri)\n    if match:\n        return {\n            'bucket': match.group(1),\n            'key': match.group(2)\n        }\n    return None\n\n# Test\nprint(parse_s3_uri('s3://my-bucket/path/to/object.txt'))\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket URI and extract the bucket name and object key from a string like 's3://my-bucket/path/to/object.txt'",
      "format": "regex"
    },
    "model_score": 5,
    "syntax_score": 10,
    "average_score": 7.5,
    "reasoning": "The solution demonstrates basic understanding of regex and URI parsing but has significant gaps in handling AWS S3 URI specifications and edge cases. The regex is overly restrictive (lowercase-only buckets) and permissive in other ways (no length validation). The silent None return and lack of input validation would cause issues in production. The solution works for the specific test case but